# Production VGG Transfer Training

Single-subject direct flatmap baseline training for BrainIT VGG low-level tokens.


## 1. Parameter/Variable/Path Definitions


In [13]:
RUN_SUBJECTS = ["subj01"]
FINAL_EVAL_IMAGES_PER_SUBJECT = 48  # Set to 1000 for the full shared-1000 final eval set.
MODEL_WEIGHTS_TIMESTAMP = "20260907_013858"
SEED = 42


In [10]:
# Modal Volume setup. Data is mounted and read like local files.
import os, re, json, random, time, gc, sys, subprocess, importlib.util, importlib.metadata, math
from collections import defaultdict, OrderedDict

DATA_DIR = "/mnt/braid2-flatmap-data"
print("data dir:", DATA_DIR)
print("data dir exists:", os.path.isdir(DATA_DIR))
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"Mount the Modal Volume at {DATA_DIR}.")


def module_available(module_name):
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

_missing = []
for module_name, package_name in [
    ("h5py", "h5py"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scipy", "scipy"),
    ("skimage", "scikit-image"),
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("tqdm", "tqdm"),
    ("matplotlib", "matplotlib"),
]:
    if not module_available(module_name):
        _missing.append(package_name)
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.io import loadmat
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from torchvision import models

for _required_knob in ["RUN_SUBJECTS", "FINAL_EVAL_IMAGES_PER_SUBJECT", "MODEL_WEIGHTS_TIMESTAMP", "SEED"]:
    if _required_knob not in globals():
        raise RuntimeError(f"Run the key knobs cell first; missing {_required_knob}")

INPUT_DIR = DATA_DIR
FLATMAP_TOKENS_DIR = f"{DATA_DIR}/flatmap_tokens_fp16"
CLIP_TARGETS_DIR = f"{DATA_DIR}/clip_targets_cache"
LOCAL_OUTPUT_ROOT = f"{DATA_DIR}/outputs"
SHARED1000_EVAL_SIZE = 1000
FINAL_EVAL_IMAGES_PER_SUBJECT = min(int(FINAL_EVAL_IMAGES_PER_SUBJECT), SHARED1000_EVAL_SIZE)
if FINAL_EVAL_IMAGES_PER_SUBJECT < 1:
    raise ValueError("FINAL_EVAL_IMAGES_PER_SUBJECT must be at least 1")

def subject_run_tag(subjects):
    return "subj" + "".join(subject[-2:] for subject in subjects)

TRAIN_SUBJECTS = list(RUN_SUBJECTS)
PREDICTION_SUBJECTS = list(RUN_SUBJECTS)
FINAL_EVAL_SUBJECTS = list(RUN_SUBJECTS)
SUBJECT_TO_IDX = {subject: i for i, subject in enumerate(TRAIN_SUBJECTS)}
IDX_TO_SUBJECT = {i: subject for subject, i in SUBJECT_TO_IDX.items()}
NUM_SUBJECTS = len(TRAIN_SUBJECTS)
TRIALS_PER_SESSION = 750
KNOWN_BAD_SESSIONS = {("subj01", 11)}

RUN_TIMESTAMP = MODEL_WEIGHTS_TIMESTAMP or time.strftime("%Y%m%d_%H%M%S")
SPLIT_POLICY = "shared1000_image_id_holdout"
VAL_FRAC = 0.05
ARCH_MODE = "baseline"
OUTPUT_RUN_NAME = f"production_vgg_subject01_baseline_{SPLIT_POLICY}_{ARCH_MODE}_{subject_run_tag(RUN_SUBJECTS)}_{RUN_TIMESTAMP}"
OUTPUT_DIR = f"{LOCAL_OUTPUT_ROOT}/{OUTPUT_RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# BrainIT low-level branch settings from paper Appendix D.1/D.2.
VGG_IMAGE_SIZE = 112
VGG_TOKEN_DIM = 512
VGG_LAYER_NAMES = ["relu1_2", "relu2_2", "relu3_3", "relu4_3", "relu5_3"]
VGG_LAYER_TOKEN_COUNTS = [56 * 56, 55 * 55, 28 * 28, 14 * 14, 7 * 7]
VGG_LAYER_OFFSETS = np.cumsum([0] + VGG_LAYER_TOKEN_COUNTS[:-1]).tolist()
VGG_TOTAL_TOKENS = int(sum(VGG_LAYER_TOKEN_COUNTS))
SAMPLE_TOKENS_PER_LAYER = [512, 512, 128, 64, 16]
TEMPERATURE = 0.07

# OpenCLIP ViT-bigG/14 internal image patch tokens, matching bridge_train.py.
CLIP_DIM = 1664
CLIP_NUM_QUERIES = 256
CLIP_LOSS_WEIGHT = 1.0
VGG_LOSS_WEIGHT = 0.25

# Batch sizes and chunk sizes.
BATCH_SIZE = 128                     # VGG+CLIP training batch size
DIP_QUERY_BATCH = 1                    # shared-1000 VGG/CLIP prediction batch size
DIP_BATCH_SIZE = 36                    # DIP inversion batch size
DIP_MIN_BATCH_SIZE = 1                 # minimum DIP batch size after OOM backoff
FINAL_DIFFUSION_BATCH_SIZE = 20        # diffusion candidate batch size, divided by ensemble size
FINAL_EVAL_FEATURE_BATCH_SIZE = 32     # final eval feature-extraction batch size
PREDICT_QUERY_CHUNK = 1024             # VGG token query chunk size during prediction


D_MODEL = 768
N_LAYERS = 4
N_HEADS = 8
FFN_DIM = 3072
DROPOUT = 0.10
EPOCHS = 10
CACHE_TRAIN_SESSIONS_IN_RAM = False
LR = 5e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 15
PLATEAU_PATIENCE = 2
PLATEAU_FACTOR = 0.1
GRAD_CLIP_NORM = 1.0
BRAIN_DIM_OVERRIDE = 768

DIP_INPUT_DIM = 32
DIP_INTERNAL_DIM = 128
DIP_SCALES = 3
DIP_STEPS = 2000
DIP_LR = 1e-2
DIP_INITIAL_NOISE = 0.1
DIP_REG_NOISE = 1.0 / 30.0
DIP_EMA = 0.99
DIP_OUTPUT_SIZE = 112
DIP_UPSAMPLED_SIZE = 256

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass
torch.backends.cudnn.benchmark = True


def ensure_xformers_wheel():
    if any(name == "xformers" or name.startswith("xformers.") for name in sys.modules):
        raise RuntimeError("Restart the kernel/container before repairing xformers; the current process already imported xformers.")

    torch_minor = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    cuda_version = torch.version.cuda
    cuda_tag = f"cu{cuda_version.replace('.', '')}" if cuda_version else None
    target_by_torch_cuda = {
        ("2.8", "12.9"): "0.0.32.post2",
    }
    target = target_by_torch_cuda.get((torch_minor, cuda_version))
    if target is None:
        raise RuntimeError(f"No xformers pin for torch {torch.__version__} / CUDA {cuda_version}; do not install xformers blindly.")

    try:
        installed = importlib.metadata.version("xformers")
    except importlib.metadata.PackageNotFoundError:
        installed = None

    if installed != target:
        print(f"installing xformers=={target} for torch {torch.__version__} / CUDA {cuda_version} (was {installed})")
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "xformers"])
        cmd = [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir", "--force-reinstall", "--no-deps",
            f"xformers=={target}",
        ]
        if cuda_tag:
            cmd += ["--index-url", f"https://download.pytorch.org/whl/{cuda_tag}"]
        subprocess.check_call(cmd)
    else:
        print(f"xformers=={target} already matches torch {torch.__version__} / CUDA {cuda_version}")

    subprocess.check_call([
        sys.executable, "-c",
        "import torch, xformers, xformers.ops; "
        "print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'xformers': xformers.__version__})",
    ])


ensure_xformers_wheel()


def torch_load_cpu(path, *args, **kwargs):
    kwargs.setdefault("map_location", "cpu")
    try:
        kwargs.setdefault("weights_only", False)
        return torch.load(path, *args, **kwargs)
    except TypeError:
        kwargs.pop("weights_only", None)
        return torch.load(path, *args, **kwargs)


def require_file(path):
    if not os.path.isfile(path):
        raise FileNotFoundError(path)
    return path

print(f"device={DEVICE} batch_size={BATCH_SIZE} output={OUTPUT_DIR}")
print(f"subjects={TRAIN_SUBJECTS}")
print(f"final_eval_images_per_subject={FINAL_EVAL_IMAGES_PER_SUBJECT} / {SHARED1000_EVAL_SIZE}")
print(f"model_weights_timestamp={MODEL_WEIGHTS_TIMESTAMP or '<new run>'}")
print(f"VGG tokens={VGG_TOTAL_TOKENS} layer_counts={VGG_LAYER_TOKEN_COUNTS}")


data dir: /mnt/braid2-flatmap-data
data dir exists: True
xformers==0.0.32.post2 already matches torch 2.8.0+cu129 / CUDA 12.9
{'torch': '2.8.0+cu129', 'cuda': '12.9', 'xformers': '0.0.32.post2'}
device=cuda batch_size=128 output=/mnt/braid2-flatmap-data/outputs/production_vgg_subject01_baseline_shared1000_image_id_holdout_baseline_subj01_20260907_013858
subjects=['subj01']
final_eval_images_per_subject=48 / 1000
model_weights_timestamp=20260907_013858
VGG tokens=7190 layer_counts=[3136, 3025, 784, 196, 49]


## 2. Setup Code/All Data Loading


In [11]:
EXP = require_file(f"{INPUT_DIR}/nsd_expdesign.mat")
mat = loadmat(EXP)

masterordering = mat["masterordering"].reshape(-1).astype(np.int64) - 1
subjectim = mat["subjectim"].astype(np.int64) - 1
shared_ids = set(mat["sharedix"].reshape(-1).astype(np.int64) - 1)

imgbrick_ids_by_subject = {}
for subject in TRAIN_SUBJECTS:
    subject_idx = int(subject[-2:]) - 1
    imgbrick_ids_by_subject[subject] = subjectim[subject_idx, masterordering]


def flatmap_candidates(subject, sess):
    return [
        f"{FLATMAP_TOKENS_DIR}/{subject}_session{sess:02d}_fp16.pt",
        f"{FLATMAP_TOKENS_DIR}/{subject}_flatmap_tokens_session{sess:02d}.pt",
    ]


def flatmap_read_path(subject, sess):
    for path in flatmap_candidates(subject, sess):
        if os.path.isfile(path):
            return path
    raise FileNotFoundError(f"Missing flatmap session {subject} {sess:02d}. Checked: {flatmap_candidates(subject, sess)}")


def discover_sessions(subject):
    if not os.path.isdir(FLATMAP_TOKENS_DIR):
        raise FileNotFoundError(FLATMAP_TOKENS_DIR)
    nums = []
    for name in os.listdir(FLATMAP_TOKENS_DIR):
        if not name.startswith(subject):
            continue
        match = re.search(r"session(\d+)", name)
        if match:
            nums.append(int(match.group(1)))
    nums = sorted(set(s for s in nums if (subject, s) not in KNOWN_BAD_SESSIONS))
    if len(nums) < 2:
        raise RuntimeError(f"{subject} needs at least 2 usable sessions, found {nums}")
    return nums


session_nums_by_subject = {subject: discover_sessions(subject) for subject in TRAIN_SUBJECTS}
train_pairs = []
val_pairs = []
for subject in TRAIN_SUBJECTS:
    sessions = session_nums_by_subject[subject]
    train_pairs.extend((subject, sess) for sess in sessions)
    val_pairs.extend((subject, sess) for sess in sessions)
    print(f"{subject}: {len(sessions)} sessions train={sessions} val={sessions}")
print(f"total: {len(train_pairs)} train sessions, {len(val_pairs)} val sessions")


def load_flatmap_tokens(subject, sess):
    brain_tokens = torch_load_cpu(flatmap_read_path(subject, sess))
    if brain_tokens.dtype != torch.float16:
        brain_tokens = brain_tokens.half()
    if not torch.isfinite(brain_tokens).all():
        bad = int((~torch.isfinite(brain_tokens)).sum().item())
        raise ValueError(f"{subject} session {sess:02d} has {bad} non-finite flatmap token values")
    return brain_tokens.contiguous()


def ids_for_session(subject, sess, n_rows):
    start = (sess - 1) * TRIALS_PER_SESSION
    return imgbrick_ids_by_subject[subject][start:start + n_rows].astype(np.int64)


FINAL_HELDOUT_IMG_IDS = set(int(i) for i in shared_ids)
VAL_HELDOUT_IMG_IDS_BY_SUBJECT = {}
for subject in TRAIN_SUBJECTS:
    nonshared_img_ids = np.array(sorted({
        int(img_id)
        for sess in session_nums_by_subject[subject]
        for img_id in ids_for_session(subject, sess, TRIALS_PER_SESSION)
        if int(img_id) not in FINAL_HELDOUT_IMG_IDS
    }), dtype=np.int64)
    image_perm = np.random.RandomState(SEED + SUBJECT_TO_IDX[subject]).permutation(nonshared_img_ids)
    n_val_images = int(VAL_FRAC * len(image_perm))
    VAL_HELDOUT_IMG_IDS_BY_SUBJECT[subject] = set(map(int, image_perm[:n_val_images]))
VAL_HELDOUT_IMG_IDS_ALL = set().union(*VAL_HELDOUT_IMG_IDS_BY_SUBJECT.values())
TRAIN_BLOCKED_IMG_IDS_BY_SUBJECT = {
    subject: FINAL_HELDOUT_IMG_IDS | VAL_HELDOUT_IMG_IDS_ALL
    for subject in TRAIN_SUBJECTS
}


def split_keep_mask(subject, img_ids, split):
    ids = [int(i) for i in img_ids.tolist()]
    if split == "train":
        blocked = TRAIN_BLOCKED_IMG_IDS_BY_SUBJECT[subject]
        keep = [img_id not in blocked for img_id in ids]
    elif split == "val":
        heldout = VAL_HELDOUT_IMG_IDS_BY_SUBJECT[subject]
        keep = [img_id in heldout for img_id in ids]
    elif split in {"all", "eval"}:
        keep = [True] * len(ids)
    else:
        raise ValueError(f"unknown split={split!r}")
    return torch.as_tensor(keep, dtype=torch.bool)


def apply_split_filter(subject, brain_tokens, img_ids, split):
    keep = split_keep_mask(subject, img_ids, split)
    return brain_tokens[keep], img_ids[keep]


def assert_clean_image_id_splits():
    for subject in TRAIN_SUBJECTS:
        train_seen = set()
        for train_subject, sess in train_pairs:
            if train_subject != subject:
                continue
            ids = torch.from_numpy(ids_for_session(subject, sess, TRIALS_PER_SESSION)).long()
            keep = split_keep_mask(subject, ids, "train")
            train_seen.update(int(i) for i in ids[keep].tolist())
        val_seen = set()
        for val_subject, sess in val_pairs:
            if val_subject != subject:
                continue
            ids = torch.from_numpy(ids_for_session(subject, sess, TRIALS_PER_SESSION)).long()
            keep = split_keep_mask(subject, ids, "val")
            val_seen.update(int(i) for i in ids[keep].tolist())
        assert train_seen.isdisjoint(FINAL_HELDOUT_IMG_IDS), f"{subject}: train/final image-id overlap"
        assert train_seen.isdisjoint(VAL_HELDOUT_IMG_IDS_ALL), f"{subject}: train/global-val image-id overlap"
        assert val_seen.isdisjoint(FINAL_HELDOUT_IMG_IDS), f"{subject}: val/final image-id overlap"
        assert train_seen.isdisjoint(val_seen), f"{subject}: train/val image-id overlap"
        print(
            f"{subject}: clean image-id split train_unique={len(train_seen)} "
            f"val_unique={len(val_seen)} final_heldout={len(FINAL_HELDOUT_IMG_IDS)}"
        )


assert_clean_image_id_splits()


def infer_flatmap_shape():
    subject, sess = train_pairs[0]
    sample = load_flatmap_tokens(subject, sess)
    if sample.ndim != 3:
        raise ValueError(f"expected flatmap tokens [trials, patches, dim], got {tuple(sample.shape)}")
    return int(sample.shape[1]), int(sample.shape[2])


NUM_FLATMAP_PATCHES, FLATMAP_TOKEN_DIM = infer_flatmap_shape()
print(f"flatmap direct tokens: patches={NUM_FLATMAP_PATCHES} dim={FLATMAP_TOKEN_DIM}")


subj01: 39 sessions train=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40] val=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
total: 39 train sessions, 39 val sessions
subj01: clean image-id split train_unique=8528 val_unique=448 final_heldout=1000
flatmap direct tokens: patches=1456 dim=768


## 3. Image Loader + BrainIT VGG Tokenizer


In [15]:
STIM_PATH = require_file(f"{INPUT_DIR}/nsd_stimuli.hdf5")
_H5 = None
_IMG_DS = None


def stimulus_dataset():
    global _H5, _IMG_DS
    if _IMG_DS is None:
        _H5 = h5py.File(STIM_PATH, "r")
        for key in ["imgBrick", "images", "stimuli"]:
            if key in _H5:
                _IMG_DS = _H5[key]
                break
        if _IMG_DS is None:
            raise KeyError(f"Could not find stimulus image dataset. HDF5 keys={list(_H5.keys())}")
    return _IMG_DS


def preprocess_images_uint8(images_np, size):
    if images_np.ndim != 4:
        raise ValueError(f"expected NHWC or NCHW images, got {images_np.shape}")
    if images_np.shape[-1] == 3:
        img = torch.from_numpy(images_np).permute(0, 3, 1, 2)
    elif images_np.shape[1] == 3:
        img = torch.from_numpy(images_np)
    else:
        raise ValueError(f"could not infer image channels from {images_np.shape}")
    img = img.float()
    if img.max() > 2:
        img = img / 255.0
    return F.interpolate(img, size=(size, size), mode="bicubic", align_corners=False).clamp(0, 1)


def images_for_ids(img_ids, size=VGG_IMAGE_SIZE):
    ids = np.asarray([int(i) for i in img_ids], dtype=np.int64)
    unique_ids, inverse = np.unique(ids, return_inverse=True)
    imgs_unique = stimulus_dataset()[unique_ids]
    imgs = imgs_unique[inverse]
    return preprocess_images_uint8(imgs, size)


class BrainITVGGTokenizer(nn.Module):
    """VGG16-BN feature maps and tokenization described in BrainIT Appendix D.2."""
    def __init__(self):
        super().__init__()
        weights = models.VGG16_BN_Weights.DEFAULT
        self.features = models.vgg16_bn(weights=weights).features.eval()
        self.capture = {5: 0, 12: 1, 22: 2, 32: 3, 42: 4}
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        for p in self.parameters():
            p.requires_grad_(False)

    @staticmethod
    def _repeat_to_512(tokens):
        if tokens.shape[-1] == VGG_TOKEN_DIM:
            return tokens
        reps = math.ceil(VGG_TOKEN_DIM / tokens.shape[-1])
        return tokens.repeat(1, 1, reps)[..., :VGG_TOKEN_DIM]

    def extract_features(self, images01):
        x = F.interpolate(images01, size=(VGG_IMAGE_SIZE, VGG_IMAGE_SIZE), mode="bicubic", align_corners=False).clamp(0, 1)
        x = (x - self.mean) / self.std
        outs = [None] * len(VGG_LAYER_NAMES)
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i in self.capture:
                outs[self.capture[i]] = x
        return outs

    def _tokens_from_feature(self, feat, layer_idx):
        if layer_idx == 0:
            tokens = F.unfold(feat, kernel_size=2, stride=2).transpose(1, 2)
        elif layer_idx == 1:
            tokens = F.unfold(feat, kernel_size=2, stride=1).transpose(1, 2)
        else:
            tokens = feat.flatten(2).transpose(1, 2)
        return self._repeat_to_512(tokens).contiguous()

    def tokens_from_features(self, features):
        return [self._tokens_from_feature(feat, i) for i, feat in enumerate(features)]

    def forward(self, images01):
        return self.tokens_from_features(self.extract_features(images01))

vgg_tokenizer = BrainITVGGTokenizer().to(DEVICE).eval()
print("VGG tokenizer ready:", VGG_LAYER_NAMES)


VGG tokenizer ready: ['relu1_2', 'relu2_2', 'relu3_3', 'relu4_3', 'relu5_3']


## 4. Define Full Model Architecture


In [16]:

class CrossAttentionBridge(nn.Module):
    def __init__(self, brain_dim, target_dim, num_queries, d_model, n_layers, n_heads, ffn_dim, dropout):
        super().__init__()
        self.query_embed = nn.Parameter(torch.randn(num_queries, d_model) * 0.02)
        self.brain_proj = nn.Linear(brain_dim, d_model) if brain_dim != d_model else nn.Identity()
        self.brain_norm = nn.LayerNorm(d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, target_dim)

    def forward(self, brain_tokens, query_indices=None):
        b = brain_tokens.shape[0]
        memory = self.brain_norm(self.brain_proj(brain_tokens))
        if query_indices is None:
            queries = self.query_embed.unsqueeze(0).expand(b, -1, -1)
        elif query_indices.ndim == 1:
            queries = self.query_embed[query_indices].unsqueeze(0).expand(b, -1, -1)
        else:
            queries = self.query_embed[query_indices]
        out = self.decoder(tgt=queries, memory=memory)
        return self.output_proj(out)


class FlatmapSingleHeadBridge(nn.Module):
    def __init__(self, brain_dim, target):
        super().__init__()
        if target not in {"vgg", "clip"}:
            raise ValueError(f"target must be 'vgg' or 'clip', got {target!r}")
        self.target = target
        self.num_patches = int(NUM_FLATMAP_PATCHES)
        if target == "vgg":
            target_dim = VGG_TOKEN_DIM
            num_queries = VGG_TOTAL_TOKENS
        else:
            target_dim = CLIP_DIM
            num_queries = CLIP_NUM_QUERIES
        self.bridge = CrossAttentionBridge(
            brain_dim=brain_dim,
            target_dim=target_dim,
            num_queries=num_queries,
            d_model=D_MODEL,
            n_layers=N_LAYERS,
            n_heads=N_HEADS,
            ffn_dim=FFN_DIM,
            dropout=DROPOUT,
        )

    def forward(self, brain_tokens, query_indices=None):
        return self.bridge(brain_tokens, query_indices=query_indices)


model = None
brain_dim = int(FLATMAP_TOKEN_DIM)


def initialize_fresh_model(target, label=None):
    global model, brain_dim
    if target not in {"vgg", "clip"}:
        raise ValueError(f"target must be 'vgg' or 'clip', got {target!r}")
    brain_dim = int(FLATMAP_TOKEN_DIM)
    model = FlatmapSingleHeadBridge(brain_dim, target=target).to(DEVICE)
    params_m = sum(p.numel() for p in model.parameters()) / 1e6
    label = label or target
    if target == "vgg":
        target_desc = f"VGG [{VGG_TOTAL_TOKENS}, {VGG_TOKEN_DIM}]"
    else:
        target_desc = f"CLIP [{CLIP_NUM_QUERIES}, {CLIP_DIM}]"
    print(f"initialized {label}: brain_dim={brain_dim} target={target_desc} params={params_m:.1f}M")
    print(f"single-subject direct flatmap baseline architecture: {NUM_FLATMAP_PATCHES} patches -> transformer memory tokens")
    return model


## 5. Train VGG Token Predictor


In [18]:
# Shared training helpers. VGG and CLIP train as separate single-head models from fresh initialization.

def sample_layer_indices(train=True):
    query_ids, splits = [], []
    for offset, n_tokens, k in zip(VGG_LAYER_OFFSETS, VGG_LAYER_TOKEN_COUNTS, SAMPLE_TOKENS_PER_LAYER):
        k = min(k, n_tokens)
        if train:
            ids = torch.randperm(n_tokens, device=DEVICE)[:k]
        else:
            ids = torch.linspace(0, n_tokens - 1, steps=k, device=DEVICE).long()
        query_ids.append(ids + offset)
        splits.append(k)
    return torch.cat(query_ids), splits


def select_target_tokens(tokens_by_layer, query_ids, splits):
    chunks = []
    start = 0
    for li, (offset, k) in enumerate(zip(VGG_LAYER_OFFSETS, splits)):
        local_ids = query_ids[start:start + k] - offset
        chunks.append(tokens_by_layer[li][:, local_ids])
        start += k
    return torch.cat(chunks, dim=1)


def info_nce_by_position(pred, target):
    b, k, _ = pred.shape
    if b < 2:
        return F.mse_loss(pred, target)
    pred = F.normalize(pred.float(), dim=-1)
    target = F.normalize(target.float(), dim=-1)
    logits = torch.einsum("bkd,ckd->kbc", pred, target) / TEMPERATURE
    labels = torch.arange(b, device=pred.device).repeat(k)
    loss_i = F.cross_entropy(logits.reshape(k * b, b), labels)
    loss_t = F.cross_entropy(logits.transpose(1, 2).reshape(k * b, b), labels)
    return 0.5 * (loss_i + loss_t)


def split_layers(x, splits):
    out, start = [], 0
    for k in splits:
        out.append(x[:, start:start + k])
        start += k
    return out


def vgg_contrastive_loss(pred, target, splits):
    loss = pred.new_tensor(0.0)
    parts = {}
    for name, p, t in zip(VGG_LAYER_NAMES, split_layers(pred, splits), split_layers(target, splits)):
        layer_loss = info_nce_by_position(p, t)
        parts[f"nce_{name}"] = layer_loss
        loss = loss + layer_loss
    return loss / len(splits), parts


def feature_metrics(pred, target):
    return {
        "mse": float(F.mse_loss(pred.float(), target.float()).detach().cpu()),
        "cos": float(F.cosine_similarity(pred.float().flatten(1), target.float().flatten(1), dim=1).mean().detach().cpu()),
    }


clip_targets_by_subject = {}


def get_clip_targets(subject):
    if subject not in clip_targets_by_subject:
        path = require_file(f"{CLIP_TARGETS_DIR}/{subject}_clip_targets.pt")
        clip_targets_by_subject[subject] = torch_load_cpu(path)
        print(f"loaded CLIP targets {subject}: {len(clip_targets_by_subject[subject])} images")
    return clip_targets_by_subject[subject]


def clip_targets_for_ids(subject, img_ids):
    clip_by_imgbrick = get_clip_targets(subject)
    targets = torch.stack([clip_by_imgbrick[int(img_id)] for img_id in img_ids]).contiguous()
    expected = (CLIP_NUM_QUERIES, CLIP_DIM)
    if tuple(targets.shape[1:]) != expected:
        raise ValueError(f"expected CLIP targets [B, {expected[0]}, {expected[1]}], got {tuple(targets.shape)}")
    if not torch.isfinite(targets).all():
        bad = int((~torch.isfinite(targets)).sum().item())
        raise ValueError(f"CLIP target batch has {bad} non-finite values")
    return targets


session_cache = {}
bad_pairs = set()


def load_session(subject, sess, split="train"):
    key = (subject, int(sess), split)
    if CACHE_TRAIN_SESSIONS_IN_RAM and key in session_cache:
        return session_cache[key]
    bad_key = (subject, int(sess))
    if bad_key in bad_pairs:
        return None
    try:
        brain_tokens = load_flatmap_tokens(subject, sess)
        ids = torch.from_numpy(ids_for_session(subject, sess, brain_tokens.shape[0])).long()
        brain_tokens, ids = apply_split_filter(subject, brain_tokens, ids, split)
        if brain_tokens.shape[0] == 0:
            return None
        loaded = (brain_tokens, ids)
        if CACHE_TRAIN_SESSIONS_IN_RAM:
            session_cache[key] = loaded
        return loaded
    except Exception as exc:
        print(f"[bad session] {subject} session {sess:02d}: {exc}; skipping")
        bad_pairs.add(bad_key)
        return None


def assert_checkpoint_split_policy(ckpt, path):
    if not isinstance(ckpt, dict):
        raise ValueError(f"refusing to load legacy bare checkpoint without split metadata: {path}")
    if ckpt.get("split_policy") != SPLIT_POLICY:
        raise ValueError(
            f"refusing to load checkpoint with split_policy={ckpt.get('split_policy')!r}; "
            f"expected {SPLIT_POLICY!r}: {path}"
        )
    if float(ckpt.get("val_frac", VAL_FRAC)) != float(VAL_FRAC):
        raise ValueError(
            f"refusing to load checkpoint with val_frac={ckpt.get('val_frac')!r}; "
            f"expected {VAL_FRAC!r}: {path}"
        )

    expected_arch_token = "direct_flatmap"
    arch = str(ckpt.get("architecture", ""))
    if expected_arch_token not in arch:
        raise ValueError(
            f"refusing to load checkpoint with architecture={arch!r}; "
            f"expected architecture containing {expected_arch_token!r}: {path}"
        )


In [20]:
# Domain-specific training helper functions.
# Run this once before the VGG and CLIP training cells.
def optimizer_lr(opt):
    return float(opt.param_groups[0]["lr"])


def set_domain_epoch_lr(opt, base_lr, epoch):
    if WARMUP_EPOCHS <= 0 or epoch > WARMUP_EPOCHS:
        return
    scale = max(epoch, 1) / WARMUP_EPOCHS
    for group in opt.param_groups:
        group["lr"] = base_lr * scale


def reset_all_trainable():
    for p in model.parameters():
        p.requires_grad_(True)


def domain_trainable_params(domain):
    if domain not in {"vgg", "clip"}:
        raise ValueError(f"unknown domain={domain!r}")
    if getattr(model, "target", None) != domain:
        raise ValueError(f"active model target={getattr(model, 'target', None)!r}; expected {domain!r}")
    reset_all_trainable()
    params = [p for p in model.parameters() if p.requires_grad]
    print(f"{domain} single-head training params: {sum(p.numel() for p in params) / 1e6:.1f}M")
    return params


def load_model_state(path):
    ckpt = torch_load_cpu(path, map_location=DEVICE)
    assert_checkpoint_split_policy(ckpt, path)
    if isinstance(ckpt, dict) and ckpt.get("domain") != getattr(model, "target", None):
        raise ValueError(f"checkpoint domain={ckpt.get('domain')!r} does not match active model target={getattr(model, 'target', None)!r}: {path}")
    model.load_state_dict(ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt)
    return ckpt


def domain_checkpoint_payload(domain, epoch, best_val, opt, scheduler, target_epochs):
    return {
        "model": model.state_dict(),
        "optimizer": opt.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": int(epoch),
        "best_val": float(best_val),
        "domain": domain,
        "source_checkpoint": None,
        "initialization": "scratch",
        "architecture": "domain_specialized_" + domain + "_single_head_from_scratch_subject01_baseline_direct_flatmap",
        "subjects": TRAIN_SUBJECTS,
        "num_flatmap_patches": NUM_FLATMAP_PATCHES,
        "architecture_mode": ARCH_MODE,
        "d_model": D_MODEL,
        "n_layers": N_LAYERS,
        "train_epochs": int(target_epochs),
        "train_lr": DOMAIN_TRAIN_LR,
        "split_policy": SPLIT_POLICY,
        "val_frac": VAL_FRAC,
        "final_heldout_image_ids_count": len(FINAL_HELDOUT_IMG_IDS),
        "val_heldout_image_ids_by_subject_count": {subject: len(ids) for subject, ids in VAL_HELDOUT_IMG_IDS_BY_SUBJECT.items()},
        "val_heldout_image_ids_all_count": len(VAL_HELDOUT_IMG_IDS_ALL),
    }


def load_domain_start(domain, latest_path, history_path, opt, scheduler):
    history = []
    best_val = float("inf")
    start_epoch = 1
    if os.path.exists(history_path):
        with open(history_path) as f:
            history = json.load(f)
        if history:
            best_val = min(h["val_loss"] for h in history)
            start_epoch = max(h["epoch"] for h in history) + 1
    if os.path.exists(latest_path):
        ckpt = load_model_state(latest_path)
        if isinstance(ckpt, dict) and "optimizer" in ckpt:
            opt.load_state_dict(ckpt["optimizer"])
        if isinstance(ckpt, dict) and "scheduler" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler"])
        if isinstance(ckpt, dict) and "epoch" in ckpt:
            start_epoch = max(start_epoch, int(ckpt["epoch"]) + 1)
        if isinstance(ckpt, dict) and "best_val" in ckpt:
            best_val = min(best_val, float(ckpt["best_val"]))
        print(f"resuming {domain} training from {latest_path}; next epoch={start_epoch}")
    else:
        print(f"starting {domain} training from fresh random initialization")
    return history, best_val, start_epoch


def save_domain_progress(domain, epoch, best_val, history, opt, scheduler, latest_path, best_path, history_path, history_csv_path, target_epochs, is_best=False):
    payload = domain_checkpoint_payload(domain, epoch, best_val, opt, scheduler, target_epochs)
    torch.save(payload, latest_path)
    if is_best:
        torch.save(payload, best_path)
    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)
    if history:
        pd.DataFrame(history).to_csv(history_csv_path, index=False)

def expected_batches_for_pairs(pairs, split):
    total = 0
    for subject, sess in pairs:
        ids = torch.from_numpy(ids_for_session(subject, sess, TRIALS_PER_SESSION)).long()
        keep = split_keep_mask(subject, ids, split)
        n_rows = int(keep.sum().item())
        if n_rows:
            total += math.ceil(n_rows / BATCH_SIZE)
    return total


def run_vgg_train_epoch(pairs, epoch, train=True):
    model.train(train)
    totals = defaultdict(float)
    n_batches = 0
    order = list(pairs)
    split = "train" if train else "val"
    if train:
        random.shuffle(order)
        set_domain_epoch_lr(vgg_train_optimizer, DOMAIN_TRAIN_LR, epoch)
    desc = f"[train vgg] epoch {epoch}/{VGG_TRAIN_EPOCHS} ({split})"
    total_batches = expected_batches_for_pairs(order, split)
    progress = tqdm(total=total_batches, desc=desc, leave=False, unit="batch")
    try:
        for subject, sess in order:
            loaded = load_session(subject, sess, split=split)
            if loaded is None:
                continue
            brain_tokens, img_ids = loaded
            perm = torch.randperm(brain_tokens.shape[0]) if train else torch.arange(brain_tokens.shape[0])
            for b0 in range(0, len(perm), BATCH_SIZE):
                idx = perm[b0:b0 + BATCH_SIZE]
                brain_batch = brain_tokens[idx].to(DEVICE, non_blocking=True).float()
                id_batch = img_ids[idx]
                images = images_for_ids(id_batch, size=VGG_IMAGE_SIZE).to(DEVICE, non_blocking=True)
                with torch.no_grad():
                    target_layers = vgg_tokenizer(images)
                query_ids, splits = sample_layer_indices(train=train)
                target = select_target_tokens(target_layers, query_ids, splits)

                with torch.set_grad_enabled(train):
                    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                        pred = model(brain_batch, query_indices=query_ids)
                        loss, loss_parts = vgg_contrastive_loss(pred, target, splits)

                if train:
                    vgg_train_optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), GRAD_CLIP_NORM)
                    vgg_train_optimizer.step()

                metrics = feature_metrics(pred, target)
                totals["loss"] += float(loss.detach().cpu())
                totals["mse"] += metrics["mse"]
                totals["cos"] += metrics["cos"]
                totals["lr"] += optimizer_lr(vgg_train_optimizer)
                for key, value in loss_parts.items():
                    totals[key] += float(value.detach().cpu())
                n_batches += 1
                progress.update(1)
                progress.set_postfix(subject=subject, sess=int(sess), loss=f"{float(loss.detach().cpu()):.4f}")
            if not CACHE_TRAIN_SESSIONS_IN_RAM:
                del brain_tokens, img_ids
                gc.collect()
    finally:
        progress.close()
    if n_batches == 0:
        raise RuntimeError(f"no valid batches for VGG training epoch; bad_pairs={sorted(bad_pairs)}")
    return {k: v / n_batches for k, v in totals.items()}



def _first_debug_batch(pairs, split="train", batch_size=None):
    batch_size = int(batch_size or min(BATCH_SIZE, 64))
    for subject, sess in pairs:
        loaded = load_session(subject, sess, split=split)
        if loaded is None:
            continue
        brain_tokens, img_ids = loaded
        if brain_tokens.shape[0] < 2:
            continue
        n = min(batch_size, brain_tokens.shape[0])
        return subject, int(sess), brain_tokens[:n], img_ids[:n]
    raise RuntimeError(f"no usable debug batch for split={split!r}")


def audit_vgg_training_batch(pairs, opt, batch_size=None):
    subject, sess, brain_batch_cpu, id_batch = _first_debug_batch(pairs, split="train", batch_size=batch_size)
    brain_batch = brain_batch_cpu.to(DEVICE, non_blocking=True).float()
    images = images_for_ids(id_batch, size=VGG_IMAGE_SIZE).to(DEVICE, non_blocking=True)
    with torch.no_grad():
        target_layers = vgg_tokenizer(images)
    query_ids, splits = sample_layer_indices(train=True)
    target = select_target_tokens(target_layers, query_ids, splits)
    opt.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        pred = model(brain_batch, query_indices=query_ids)
        loss, loss_parts = vgg_contrastive_loss(pred, target, splits)
    loss.backward()
    grad_sq = 0.0
    grad_params = 0
    for p in model.parameters():
        if p.grad is not None:
            grad_sq += float(p.grad.detach().float().square().sum().cpu())
            grad_params += 1
    opt.zero_grad(set_to_none=True)
    metrics = feature_metrics(pred.detach(), target.detach())
    chance = math.log(max(int(len(id_batch)), 2))
    print(
        f"VGG batch audit subject={subject} sess={sess} batch={len(id_batch)} unique_ids={len(set(int(i) for i in id_batch.tolist()))} "
        f"loss={float(loss.detach().cpu()):.4f} chance_log_batch={chance:.4f} cos={metrics['cos']:.4f} "
        f"pred_std={float(pred.detach().float().std().cpu()):.4f} target_std={float(target.detach().float().std().cpu()):.4f} "
        f"grad_norm={math.sqrt(grad_sq):.4e} grad_tensors={grad_params}"
    )
    return {"loss": float(loss.detach().cpu()), "cos": metrics["cos"], "grad_norm": math.sqrt(grad_sq), "grad_tensors": grad_params}


def debug_overfit_vgg_one_batch(steps=50, batch_size=16, lr=1e-3):
    subject, sess, brain_batch_cpu, id_batch = _first_debug_batch(train_pairs, split="train", batch_size=batch_size)
    state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    brain_batch = brain_batch_cpu.to(DEVICE, non_blocking=True).float()
    images = images_for_ids(id_batch, size=VGG_IMAGE_SIZE).to(DEVICE, non_blocking=True)
    with torch.no_grad():
        target_layers = vgg_tokenizer(images)
    query_ids, splits = sample_layer_indices(train=False)
    target = select_target_tokens(target_layers, query_ids, splits)
    losses = []
    model.train(True)
    for step in range(1, int(steps) + 1):
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            pred = model(brain_batch, query_indices=query_ids)
            loss, _ = vgg_contrastive_loss(pred, target, splits)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        opt.step()
        losses.append(float(loss.detach().cpu()))
    model.load_state_dict(state)
    model.to(DEVICE)
    print(f"VGG one-batch overfit debug subject={subject} sess={sess} steps={steps} batch={len(id_batch)} loss {losses[0]:.4f} -> {losses[-1]:.4f}; model restored")
    return losses


In [ ]:
# Set this to the total VGG training epoch you want this cell to reach.
# Example: leave at 10 to skip after epoch 10; change to 20 to train epochs 11-20.
VGG_TRAIN_EPOCHS = 30

# Domain-specific training: VGG head from scratch.
# Resumes from vgg_latest.pt when present; otherwise starts from fresh random initialization.
# Optional: set this to an existing run timestamp here so you can start rerunning at this cell.
TRAIN_MODEL_WEIGHTS_TIMESTAMP = globals().get("MODEL_WEIGHTS_TIMESTAMP", "")
if TRAIN_MODEL_WEIGHTS_TIMESTAMP:
    RUN_TIMESTAMP = TRAIN_MODEL_WEIGHTS_TIMESTAMP
    OUTPUT_RUN_NAME = f"production_vgg_subject01_baseline_{SPLIT_POLICY}_{ARCH_MODE}_{subject_run_tag(RUN_SUBJECTS)}_{RUN_TIMESTAMP}"
    OUTPUT_DIR = f"{LOCAL_OUTPUT_ROOT}/{OUTPUT_RUN_NAME}"

DOMAIN_TRAIN_LR = LR

BEST_VGG_CKPT_PATH = f"{OUTPUT_DIR}/vgg_best.pt"
LATEST_VGG_CKPT_PATH = f"{OUTPUT_DIR}/vgg_latest.pt"
BEST_CLIP_CKPT_PATH = f"{OUTPUT_DIR}/clip_best.pt"
LATEST_CLIP_CKPT_PATH = f"{OUTPUT_DIR}/clip_latest.pt"
VGG_TRAIN_HISTORY_PATH = f"{OUTPUT_DIR}/train_vgg_history.json"
VGG_TRAIN_HISTORY_CSV_PATH = f"{OUTPUT_DIR}/train_vgg_history.csv"
CLIP_TRAIN_HISTORY_PATH = f"{OUTPUT_DIR}/train_clip_history.json"
CLIP_TRAIN_HISTORY_CSV_PATH = f"{OUTPUT_DIR}/train_clip_history.csv"

initialize_fresh_model("vgg", "vgg scratch")

vgg_train_optimizer = torch.optim.AdamW(domain_trainable_params("vgg"), lr=DOMAIN_TRAIN_LR, weight_decay=WEIGHT_DECAY)
vgg_train_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    vgg_train_optimizer,
    mode="min",
    factor=PLATEAU_FACTOR,
    patience=PLATEAU_PATIENCE,
)
RUN_VGG_BATCH_AUDIT = globals().get("RUN_VGG_BATCH_AUDIT", True)

vgg_train_history, best_vgg_val, vgg_train_start_epoch = load_domain_start(
    "vgg",
    LATEST_VGG_CKPT_PATH,
    VGG_TRAIN_HISTORY_PATH,
    vgg_train_optimizer,
    vgg_train_scheduler,
)

if RUN_VGG_BATCH_AUDIT:
    audit_vgg_training_batch(train_pairs, vgg_train_optimizer)

if vgg_train_start_epoch > VGG_TRAIN_EPOCHS:
    print(f"VGG training already complete for VGG_TRAIN_EPOCHS={VGG_TRAIN_EPOCHS}; loaded {len(vgg_train_history)} rows")
else:
    for epoch in range(vgg_train_start_epoch, VGG_TRAIN_EPOCHS + 1):
        train_scores = run_vgg_train_epoch(train_pairs, epoch, train=True)
        val_scores = run_vgg_train_epoch(val_pairs, epoch, train=False)
        row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_scores.items()}, **{f"val_{k}": v for k, v in val_scores.items()}}
        vgg_train_history.append(row)
        improved = row["val_loss"] < best_vgg_val
        if improved:
            best_vgg_val = row["val_loss"]
        if epoch > WARMUP_EPOCHS:
            vgg_train_scheduler.step(row["val_loss"])
        save_domain_progress(
            "vgg",
            epoch,
            best_vgg_val,
            vgg_train_history,
            vgg_train_optimizer,
            vgg_train_scheduler,
            LATEST_VGG_CKPT_PATH,
            BEST_VGG_CKPT_PATH,
            VGG_TRAIN_HISTORY_PATH,
            VGG_TRAIN_HISTORY_CSV_PATH,
            VGG_TRAIN_EPOCHS,
            is_best=improved,
        )
        print(
            f"VGG training epoch {epoch:02d} lr={optimizer_lr(vgg_train_optimizer):.2e} "
            f"train_loss={row['train_loss']:.4f} train_cos={row['train_cos']:.4f} "
            f"val_loss={row['val_loss']:.4f} val_cos={row['val_cos']:.4f} best_val={best_vgg_val:.4f}"
        )

reset_all_trainable()


initialized vgg scratch: brain_dim=768 target=VGG [7190, 512] params=43.7M
single-subject direct flatmap baseline architecture: 1456 patches -> transformer memory tokens
vgg single-head training params: 43.7M
resuming vgg training from /mnt/braid2-flatmap-data/outputs/production_vgg_subject01_baseline_shared1000_image_id_holdout_baseline_subj01_20260907_013858/vgg_latest.pt; next epoch=26
VGG batch audit subject=subj01 sess=1 batch=64 unique_ids=62 loss=2.0627 chance_log_batch=4.1589 cos=0.1996 pred_std=1.0984 target_std=0.2464 grad_norm=6.6371e-01 grad_tensors=77


[train vgg] epoch 26/30 (train):   0%|          | 0/215 [00:00<?, ?batch/s]

[train vgg] epoch 26/30 (val):   0%|          | 0/39 [00:00<?, ?batch/s]

VGG training epoch 26 lr=5.00e-06 train_loss=2.5421 train_cos=0.1923 val_loss=2.4404 val_cos=0.1439 best_val=2.4229


[train vgg] epoch 27/30 (train):   0%|          | 0/215 [00:00<?, ?batch/s]

...

VGG training epoch 28 lr=5.00e-07 train_loss=2.5294 train_cos=0.1929 val_loss=2.4406 val_cos=0.1438 best_val=2.4229


[train vgg] epoch 29/30 (train):   0%|          | 0/215 [00:00<?, ?batch/s]

[train vgg] epoch 29/30 (val):   0%|          | 0/39 [00:00<?, ?batch/s]

VGG training epoch 29 lr=5.00e-07 train_loss=2.5349 train_cos=0.1931 val_loss=2.4405 val_cos=0.1437 best_val=2.4229


[train vgg] epoch 30/30 (train):   0%|          | 0/215 [00:00<?, ?batch/s]

## 6. Predict Shared-1000 VGG Tokens


In [ ]:
@torch.no_grad()
def predict_all_vgg_tokens(brain_batch, subject):
    model.eval()
    brain_batch = brain_batch.to(DEVICE, non_blocking=True).float()
    outs = []
    for start in range(0, VGG_TOTAL_TOKENS, PREDICT_QUERY_CHUNK):
        ids = torch.arange(start, min(start + PREDICT_QUERY_CHUNK, VGG_TOTAL_TOKENS), device=DEVICE)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            outs.append(model(brain_batch, query_indices=ids).float().cpu())
    return torch.cat(outs, dim=1)


def split_all_vgg_tokens(x):
    out, start = [], 0
    for n in VGG_LAYER_TOKEN_COUNTS:
        out.append(x[:, start:start + n])
        start += n
    return out


def shared_tokens_path(subject):
    return f"{OUTPUT_DIR}/{subject}_shared1000_flatmap_source_tokens.pt"


def shared_ids_path(subject):
    return f"{OUTPUT_DIR}/{subject}_shared1000_img_ids.json"


def pred_vgg_path(subject):
    return f"{OUTPUT_DIR}/{subject}_shared1000_transfer_pred_vgg_tokens.pt"


def collect_shared1000_flatmap_tokens(subject):
    tokens_path = shared_tokens_path(subject)
    ids_path = shared_ids_path(subject)
    if os.path.exists(tokens_path) and os.path.exists(ids_path):
        with open(ids_path) as f:
            ids = json.load(f)
        if any(int(img_id) not in FINAL_HELDOUT_IMG_IDS for img_id in ids):
            raise ValueError(f"{subject} cached shared1000 IDs include non-heldout image IDs")
        return torch_load_cpu(tokens_path), ids
    groups = defaultdict(list)
    shared_id_set = set(int(i) for i in shared_ids)
    for sess in tqdm(session_nums_by_subject[subject], desc=f"collect shared-1000 flatmaps {subject}"):
        brain_tokens = load_flatmap_tokens(subject, sess)
        sess_ids = ids_for_session(subject, sess, brain_tokens.shape[0])
        for row, img_id in enumerate(sess_ids):
            if int(img_id) in shared_id_set:
                groups[int(img_id)].append(brain_tokens[row])
        del brain_tokens
        gc.collect()
    ids = sorted(groups)
    if any(int(img_id) not in FINAL_HELDOUT_IMG_IDS for img_id in ids):
        raise ValueError(f"{subject} shared1000 collection included non-heldout image IDs")
    avg_tokens = torch.stack([torch.stack(groups[img_id]).mean(0) for img_id in ids]).half().contiguous()
    torch.save(avg_tokens, tokens_path)
    with open(ids_path, "w") as f:
        json.dump([int(i) for i in ids], f)
    return avg_tokens, ids


BEST_VGG_CKPT_PATH = globals().get("BEST_VGG_CKPT_PATH", f"{OUTPUT_DIR}/vgg_best.pt")
LATEST_VGG_CKPT_PATH = globals().get("LATEST_VGG_CKPT_PATH", f"{OUTPUT_DIR}/vgg_latest.pt")


def prediction_checkpoint_for_head(head):
    if head != "vgg":
        raise ValueError(f"production-vgg-train only predicts VGG, got head={head!r}")
    candidates = [BEST_VGG_CKPT_PATH, LATEST_VGG_CKPT_PATH]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"no checkpoint available for VGG; checked {candidates}")


def load_prediction_checkpoint(path, head="vgg"):
    initialize_fresh_model(head, f"{head} prediction")
    ckpt = torch_load_cpu(path, map_location=DEVICE)
    assert_checkpoint_split_policy(ckpt, path)
    if isinstance(ckpt, dict) and ckpt.get("domain") != head:
        raise ValueError(f"checkpoint domain={ckpt.get('domain')!r} does not match requested head={head!r}: {path}")
    model.load_state_dict(ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt)
    print("loaded prediction checkpoint:", path)


def predict_shared1000_vgg_for_subject(subject):
    shared_brain_tokens, shared_img_ids = collect_shared1000_flatmap_tokens(subject)
    print(f"{subject} shared flatmap source tokens:", tuple(shared_brain_tokens.shape), "ids:", len(shared_img_ids))
    vgg_path = pred_vgg_path(subject)
    if os.path.exists(vgg_path):
        print(f"{subject}: predicted VGG exists:", vgg_path)
        return vgg_path

    load_prediction_checkpoint(prediction_checkpoint_for_head("vgg"), "vgg")
    vgg_preds = []
    for start in tqdm(range(0, len(shared_brain_tokens), DIP_QUERY_BATCH), desc=f"predict shared-1000 VGG tokens {subject}"):
        brain_batch = shared_brain_tokens[start:start + DIP_QUERY_BATCH]
        vgg_preds.append(predict_all_vgg_tokens(brain_batch, subject))
    pred_vgg_tokens = torch.cat(vgg_preds, dim=0).half().contiguous()
    torch.save(pred_vgg_tokens, vgg_path)
    print(f"{subject}: saved predicted VGG tokens:", tuple(pred_vgg_tokens.shape), vgg_path)
    del pred_vgg_tokens, vgg_preds, shared_brain_tokens
    gc.collect()
    return vgg_path


prediction_manifest = {}
for subject in PREDICTION_SUBJECTS:
    vgg_path = predict_shared1000_vgg_for_subject(subject)
    prediction_manifest[subject] = {
        "pred_vgg_path": vgg_path,
        "shared_tokens_path": shared_tokens_path(subject),
        "shared_ids_path": shared_ids_path(subject),
    }
with open(f"{OUTPUT_DIR}/transfer_vgg_prediction_manifest.json", "w") as f:
    json.dump(prediction_manifest, f, indent=2)


def load_subject_prediction_for_lowlevel(subject):
    with open(shared_ids_path(subject)) as f:
        ids = json.load(f)
    return {
        "subject": subject,
        "shared_img_ids": ids,
        "pred_vgg_tokens": torch_load_cpu(pred_vgg_path(subject)),
    }


for subject in PREDICTION_SUBJECTS:
    subject_prediction = load_subject_prediction_for_lowlevel(subject)
    print(
        "prediction subject:", subject,
        "predicted VGG:", tuple(subject_prediction["pred_vgg_tokens"].shape),
        "ids:", len(subject_prediction["shared_img_ids"]),
    )


## 7. DIP Image Inversion + Preview


In [ ]:
class GroupedConvBlock(nn.Module):
    def __init__(self, groups, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(groups * in_ch, groups * out_ch, 3, padding=1, groups=groups),
            nn.BatchNorm2d(groups * out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(groups * out_ch, groups * out_ch, 3, padding=1, groups=groups),
            nn.BatchNorm2d(groups * out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class BatchedDIPUNet(nn.Module):
    def __init__(self, groups, input_dim=DIP_INPUT_DIM, width=DIP_INTERNAL_DIM):
        super().__init__()
        self.groups = int(groups)
        self.input_dim = int(input_dim)
        self.width = int(width)
        self.down1 = GroupedConvBlock(groups, input_dim, width)
        self.down2 = GroupedConvBlock(groups, width, width)
        self.down3 = GroupedConvBlock(groups, width, width)
        self.mid = GroupedConvBlock(groups, width, width)
        self.up3 = GroupedConvBlock(groups, width * 2, width)
        self.up2 = GroupedConvBlock(groups, width * 2, width)
        self.up1 = GroupedConvBlock(groups, width * 2, width)
        self.out = nn.Conv2d(groups * width, groups * 3, 1, groups=groups)

    def _merge(self, x):
        b, c, h, w = x.shape
        if b != self.groups:
            raise ValueError(f"expected batch={self.groups}, got {b}")
        return x.reshape(1, b * c, h, w)

    def _split(self, x, channels):
        _, _, h, w = x.shape
        return x.reshape(self.groups, channels, h, w)

    def forward(self, z):
        z = self._merge(z)
        e1 = self.down1(z)
        e2 = self.down2(F.avg_pool2d(e1, 2))
        e3 = self.down3(F.avg_pool2d(e2, 2))
        x = self.mid(F.avg_pool2d(e3, 2))
        x = F.interpolate(x, size=e3.shape[-2:], mode="bilinear", align_corners=False)
        x = self.up3(torch.cat([x, e3], dim=1))
        x = F.interpolate(x, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        x = self.up2(torch.cat([x, e2], dim=1))
        x = F.interpolate(x, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        return self._split(torch.sigmoid(self.out(x)), 3)


def reduce_repeated_token_dim(tokens, true_dim):
    if tokens.shape[-1] == true_dim:
        return tokens
    if tokens.shape[-1] % true_dim != 0:
        return tokens[..., :true_dim]
    return tokens.reshape(*tokens.shape[:-1], tokens.shape[-1] // true_dim, true_dim).mean(dim=-2)


def fold_overlapping_tokens(tokens, channels, output_hw, kernel_size, stride):
    b = tokens.shape[0]
    cols = tokens.transpose(1, 2).contiguous()
    feat = F.fold(cols, output_size=output_hw, kernel_size=kernel_size, stride=stride)
    ones = torch.ones((1, channels * kernel_size * kernel_size, tokens.shape[1]), device=tokens.device, dtype=tokens.dtype)
    counts = F.fold(ones, output_size=output_hw, kernel_size=kernel_size, stride=stride).clamp_min(1e-6)
    return feat / counts


def tokens_to_brainit_vgg_features(tokens):
    layers = split_all_vgg_tokens(tokens.float())
    feat1_tokens = reduce_repeated_token_dim(layers[0], 4 * 64)
    feat2_tokens = reduce_repeated_token_dim(layers[1], 4 * 128)
    feat3_tokens = reduce_repeated_token_dim(layers[2], 256)
    feat4_tokens = reduce_repeated_token_dim(layers[3], 512)
    feat5_tokens = reduce_repeated_token_dim(layers[4], 512)

    feat1 = fold_overlapping_tokens(feat1_tokens, channels=64, output_hw=(112, 112), kernel_size=2, stride=2)
    feat2 = fold_overlapping_tokens(feat2_tokens, channels=128, output_hw=(56, 56), kernel_size=2, stride=1)
    feat3 = feat3_tokens.transpose(1, 2).reshape(tokens.shape[0], 256, 28, 28).contiguous()
    feat4 = feat4_tokens.transpose(1, 2).reshape(tokens.shape[0], 512, 14, 14).contiguous()
    feat5 = feat5_tokens.transpose(1, 2).reshape(tokens.shape[0], 512, 7, 7).contiguous()
    return [feat1, feat2, feat3, feat4, feat5]


DIP_LAYER_WEIGHTS = [1.0, 1.0, 1.0, 1.0, 1.0]
DIP_COMPILE = False
DIP_PARALLEL_GPUS = globals().get("DIP_PARALLEL_GPUS", True)
DIP_GPU_IDS = globals().get("DIP_GPU_IDS", None)  # None means all visible CUDA devices.
DIP_BATCH_SIZE_PER_GPU = int(globals().get("DIP_BATCH_SIZE_PER_GPU", DIP_BATCH_SIZE))

import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

DIP_DATA_LOCK = threading.Lock()


def canonical_torch_device(device):
    device = torch.device(device)
    if device.type == "cuda" and device.index is None:
        return torch.device(f"cuda:{torch.cuda.current_device()}")
    return device


def dip_loss(pred_img, target_features, tokenizer=None):
    tokenizer = vgg_tokenizer if tokenizer is None else tokenizer
    pred_features = tokenizer.extract_features(pred_img)
    loss = pred_img.new_tensor(0.0)
    for weight, pred, target in zip(DIP_LAYER_WEIGHTS, pred_features, target_features):
        loss = loss + weight * F.mse_loss(pred.float(), target.float())
    return loss / sum(DIP_LAYER_WEIGHTS)


def maybe_compile_dip(dip):
    if not DIP_COMPILE or not hasattr(torch, "compile"):
        return dip
    try:
        return torch.compile(dip, mode="reduce-overhead")
    except Exception as exc:
        print(f"torch.compile skipped: {exc}")
        return dip


def invert_vgg_features_with_dip(target_features, desc="DIP", device=DEVICE, tokenizer=None, generator=None):
    batch_size = int(target_features[0].shape[0])
    device = canonical_torch_device(device)
    tokenizer = vgg_tokenizer if tokenizer is None else tokenizer
    dip = BatchedDIPUNet(batch_size).to(device).train()
    dip = maybe_compile_dip(dip)
    z = torch.randn(
        batch_size,
        DIP_INPUT_DIM,
        DIP_OUTPUT_SIZE,
        DIP_OUTPUT_SIZE,
        device=device,
        generator=generator,
    ) * DIP_INITIAL_NOISE
    opt = torch.optim.Adam(dip.parameters(), lr=DIP_LR)
    ema = None
    target_features = [t.to(device).float().detach() for t in target_features]

    pbar = tqdm(range(DIP_STEPS), desc=desc, leave=False, mininterval=2.0)
    for step in pbar:
        z_noise = torch.randn(z.shape, device=device, dtype=z.dtype, generator=generator)
        z_in = z + z_noise * DIP_REG_NOISE
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=device.type == "cuda"):
            img = dip(z_in)
            loss = dip_loss(img, target_features, tokenizer=tokenizer)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        with torch.no_grad():
            current = dip(z).detach()
            ema = current if ema is None else DIP_EMA * ema + (1.0 - DIP_EMA) * current
        if step % 100 == 0:
            pbar.set_postfix(loss=f"{float(loss.detach().cpu()):.4f}")

    lowlevel_112 = ema.clamp(0, 1)
    lowlevel_256 = F.interpolate(lowlevel_112, size=(DIP_UPSAMPLED_SIZE, DIP_UPSAMPLED_SIZE), mode="bicubic", align_corners=False).clamp(0, 1)
    return lowlevel_112.cpu(), lowlevel_256.cpu()


DIP_VERSION = "v2"
DIP_SAMPLE_SEED = SEED
RERUN_DIP_INVERSION = False
DIP_USE_GROUND_TRUTH_VGG = False

DIP_TARGET_SOURCE = "gtvgg" if DIP_USE_GROUND_TRUTH_VGG else "predvgg"
DIP_OUTPUT_TAG = f"{DIP_VERSION}_gtvgg" if DIP_USE_GROUND_TRUTH_VGG else DIP_VERSION


def dip_paths_for_subject(subject):
    recon_dir = f"{OUTPUT_DIR}/{subject}_shared1000_dip_recon_shards_{DIP_OUTPUT_TAG}"
    os.makedirs(recon_dir, exist_ok=True)
    return {
        "recon_dir": recon_dir,
        "recons_path": f"{OUTPUT_DIR}/{subject}_shared1000_dip_recons_{DIP_OUTPUT_TAG}.pt",
        "recons112_path": f"{OUTPUT_DIR}/{subject}_shared1000_dip_recons_112_{DIP_OUTPUT_TAG}.pt",
        "img_ids_path": f"{OUTPUT_DIR}/{subject}_shared1000_dip_img_ids_{DIP_OUTPUT_TAG}.json",
        "source_indices_path": f"{OUTPUT_DIR}/{subject}_shared1000_dip_source_indices_{DIP_OUTPUT_TAG}.json",
    }


def dip_shard_path(paths, i):
    return f"{paths['recon_dir']}/recon_{i:04d}.pt"


def dip_target_features_for_indices(subject, batch_indices, shared_img_ids, pred_vgg_tokens, device=DEVICE, tokenizer=None):
    device = canonical_torch_device(device)
    tokenizer = vgg_tokenizer if tokenizer is None else tokenizer
    if DIP_USE_GROUND_TRUTH_VGG:
        img_ids = [int(shared_img_ids[i]) for i in batch_indices]
        with DIP_DATA_LOCK:
            images = images_for_ids(img_ids, size=VGG_IMAGE_SIZE).to(device, non_blocking=True)
        with torch.no_grad():
            target_features = tokenizer.extract_features(images)
        target_features = [feat.detach().cpu() for feat in target_features]
        del images
        return target_features
    return tokens_to_brainit_vgg_features(pred_vgg_tokens[batch_indices])


def final_eval_indices_for_subject(subject, n_available):
    n_requested = min(FINAL_EVAL_IMAGES_PER_SUBJECT, n_available)
    if n_requested >= n_available:
        return list(range(n_available))
    seed = DIP_SAMPLE_SEED + SUBJECT_TO_IDX[subject]
    rng = np.random.RandomState(seed)
    return sorted(rng.choice(n_available, size=n_requested, replace=False).astype(int).tolist())


def run_dip_inversion_for_subject(subject, device=DEVICE, tokenizer=None):
    device = canonical_torch_device(device)
    tokenizer = vgg_tokenizer if tokenizer is None else tokenizer
    if device.type == "cuda":
        torch.cuda.set_device(device)
    dip_generator = torch.Generator(device=device)
    dip_generator.manual_seed(DIP_SAMPLE_SEED + SUBJECT_TO_IDX[subject])
    active_prediction = load_subject_prediction_for_lowlevel(subject)
    shared_img_ids = active_prediction["shared_img_ids"]
    pred_vgg_tokens = active_prediction["pred_vgg_tokens"]
    paths = dip_paths_for_subject(subject)
    print("DIP subject:", subject, "device:", str(device), "predicted VGG:", tuple(pred_vgg_tokens.shape), "ids:", len(shared_img_ids))

    n_available = min(SHARED1000_EVAL_SIZE, len(shared_img_ids), pred_vgg_tokens.shape[0])
    recon_indices = final_eval_indices_for_subject(subject, n_available)
    pending_indices = [i for i in recon_indices if RERUN_DIP_INVERSION or not os.path.exists(dip_shard_path(paths, i))]
    active_batch_size = min(DIP_BATCH_SIZE_PER_GPU, max(len(pending_indices), 1))
    print(
        f"DIP images to invert for {subject}: {len(pending_indices)} pending / {len(recon_indices)} requested / "
        f"{n_available} available, batch_size_per_gpu={active_batch_size}, rerun={RERUN_DIP_INVERSION}, target={DIP_TARGET_SOURCE}"
    )
    start = 0
    pbar = tqdm(total=len(pending_indices), desc=f"DIP invert batches {subject}")
    while start < len(pending_indices):
        batch_indices = pending_indices[start:start + active_batch_size]
        target_features = recon112 = recon256 = None
        try:
            target_features = dip_target_features_for_indices(subject, batch_indices, shared_img_ids, pred_vgg_tokens, device=device, tokenizer=tokenizer)
            recon112, recon256 = invert_vgg_features_with_dip(
                target_features,
                desc=f"DIP {subject} {DIP_TARGET_SOURCE} {device} batch {start // active_batch_size + 1}",
                device=device,
                tokenizer=tokenizer,
                generator=dip_generator,
            )
            for j, i in enumerate(batch_indices):
                torch.save({
                    "img_id": int(shared_img_ids[i]),
                    "source_index": int(i),
                    "recon112": recon112[j].half(),
                    "recon": recon256[j].half(),
                    "target_source": DIP_TARGET_SOURCE,
                }, dip_shard_path(paths, i))
            start += len(batch_indices)
            pbar.update(len(batch_indices))
            del target_features, recon112, recon256
        except RuntimeError as exc:
            del target_features, recon112, recon256
            if device.type == "cuda" and "out of memory" in str(exc).lower() and active_batch_size > DIP_MIN_BATCH_SIZE:
                active_batch_size = max(DIP_MIN_BATCH_SIZE, active_batch_size // 2)
                print(f"CUDA OOM; retrying with DIP batch_size={active_batch_size}")
            else:
                raise
        gc.collect()
        if device.type == "cuda":
            with torch.cuda.device(device):
                torch.cuda.empty_cache()
    pbar.close()

    recons112, recons256, ids_done, source_indices_done = [], [], [], []
    for i in recon_indices:
        item = torch_load_cpu(dip_shard_path(paths, i))
        ids_done.append(int(item["img_id"]))
        source_indices_done.append(int(item.get("source_index", i)))
        recons112.append(item["recon112"].float())
        recons256.append(item["recon"].float())
    all_recons112 = torch.stack(recons112).clamp(0, 1)
    all_recons = torch.stack(recons256).clamp(0, 1)
    torch.save(all_recons112, paths["recons112_path"])
    torch.save(all_recons, paths["recons_path"])
    with open(paths["img_ids_path"], "w") as f:
        json.dump(ids_done, f)
    with open(paths["source_indices_path"], "w") as f:
        json.dump(source_indices_done, f)
    print("saved DIP recons 112:", tuple(all_recons112.shape), paths["recons112_path"])
    print("saved DIP recons 256:", tuple(all_recons.shape), paths["recons_path"])
    return {"subject": subject, "n_requested": len(recon_indices), "n_available": n_available, **paths}


def dip_worker_devices():
    if DEVICE.type != "cuda" or not DIP_PARALLEL_GPUS or torch.cuda.device_count() <= 1:
        return [DEVICE]
    if DIP_GPU_IDS is None:
        gpu_ids = list(range(torch.cuda.device_count()))
    else:
        gpu_ids = [int(i) for i in DIP_GPU_IDS]
    if not gpu_ids:
        return [DEVICE]
    return [torch.device(f"cuda:{i}") for i in gpu_ids]


def make_dip_tokenizer_for_device(device):
    device = canonical_torch_device(device)
    try:
        existing_device = next(vgg_tokenizer.parameters()).device
    except StopIteration:
        existing_device = vgg_tokenizer.mean.device
    if device == existing_device:
        return vgg_tokenizer
    if device.type == "cuda":
        with torch.cuda.device(device):
            return BrainITVGGTokenizer().to(device).eval()
    return BrainITVGGTokenizer().to(device).eval()


def run_dip_subject_group(subjects, device, tokenizer):
    results = {}
    for subject in subjects:
        results[subject] = run_dip_inversion_for_subject(subject, device=device, tokenizer=tokenizer)
    return results


dip_devices = dip_worker_devices()
if len(dip_devices) == 1 or len(FINAL_EVAL_SUBJECTS) <= 1:
    dip_device = dip_devices[0]
    dip_tokenizer = make_dip_tokenizer_for_device(dip_device)
    print("DIP inversion device:", str(dip_device), "batch_size_per_gpu:", DIP_BATCH_SIZE_PER_GPU)
    dip_manifest = {
        subject: run_dip_inversion_for_subject(subject, device=dip_device, tokenizer=dip_tokenizer)
        for subject in FINAL_EVAL_SUBJECTS
    }
else:
    subject_groups = [list(FINAL_EVAL_SUBJECTS[i::len(dip_devices)]) for i in range(len(dip_devices))]
    worker_specs = [(device, group) for device, group in zip(dip_devices, subject_groups) if group]
    print(
        "DIP inversion devices:",
        {str(device): group for device, group in worker_specs},
        "batch_size_per_gpu:", DIP_BATCH_SIZE_PER_GPU,
        "effective_parallel_batch:", DIP_BATCH_SIZE_PER_GPU * len(worker_specs),
    )
    dip_tokenizers = {str(device): make_dip_tokenizer_for_device(device) for device, _ in worker_specs}
    dip_manifest = {}
    with ThreadPoolExecutor(max_workers=len(worker_specs)) as executor:
        futures = [
            executor.submit(run_dip_subject_group, group, device, dip_tokenizers[str(device)])
            for device, group in worker_specs
        ]
        for future in as_completed(futures):
            dip_manifest.update(future.result())
    dip_manifest = {subject: dip_manifest[subject] for subject in FINAL_EVAL_SUBJECTS}

DIP_MANIFEST_PATH = f"{OUTPUT_DIR}/dip_prediction_manifest_{DIP_OUTPUT_TAG}.json"
with open(DIP_MANIFEST_PATH, "w") as f:
    json.dump(dip_manifest, f, indent=2)
print("saved DIP manifest:", DIP_MANIFEST_PATH)


In [ ]:
from skimage.color import rgb2gray
from skimage.metrics import structural_similarity as ssim_fn

PREVIEW_GRID_N = 48
PREVIEW_GRID_COLS = 6


def eval_dip_subject(subject, paths):
    eval_path = f"{OUTPUT_DIR}/{subject}_shared1000_dip_eval_{DIP_OUTPUT_TAG}.csv"
    preview_path = f"{OUTPUT_DIR}/{subject}_shared1000_dip_preview_grid_{DIP_OUTPUT_TAG}.png"

    all_recons = torch_load_cpu(paths["recons_path"]).float().clamp(0, 1)
    with open(paths["img_ids_path"]) as f:
        ids_done = json.load(f)
    ids_done = ids_done[:len(all_recons)]
    true_images = images_for_ids(ids_done, size=DIP_UPSAMPLED_SIZE).float()

    pixel_mse = F.mse_loss(all_recons, true_images).item()
    pixcorr_scores, ssim_scores = [], []
    for rec, true in tqdm(list(zip(all_recons, true_images)), desc=f"eval DIP recons {subject}"):
        rec_np = rec.permute(1, 2, 0).numpy()
        true_np = true.permute(1, 2, 0).numpy()
        pixcorr_scores.append(np.corrcoef(rec_np.reshape(-1), true_np.reshape(-1))[0, 1])
        ssim_scores.append(ssim_fn(rgb2gray(true_np), rgb2gray(rec_np), data_range=1.0))

    row = {
        "subject": subject,
        "N": len(all_recons),
        "Pixel_MSE": pixel_mse,
        "PixCorr": float(np.nanmean(pixcorr_scores)),
        "SSIM": float(np.nanmean(ssim_scores)),
        "target_source": DIP_TARGET_SOURCE,
    }
    pd.DataFrame([row]).to_csv(eval_path, index=False)
    print(pd.DataFrame([row]))
    print("saved:", eval_path)

    n_show = min(PREVIEW_GRID_N, len(all_recons))
    show_idx = np.arange(n_show)
    cols = min(PREVIEW_GRID_COLS, n_show)
    rows = int(np.ceil(n_show / cols))
    combined = torch.cat([true_images[show_idx], all_recons[show_idx]], dim=3).clamp(0, 1)
    fig, axes = plt.subplots(rows, cols, figsize=(4.8 * cols, 3.0 * rows), constrained_layout=True)
    axes = np.atleast_1d(axes).reshape(rows, cols)
    for ax in axes.ravel():
        ax.axis("off")
    for k, idx in enumerate(show_idx):
        ax = axes[k // cols, k % cols]
        ax.imshow(combined[k].permute(1, 2, 0))
        ax.set_title(f"{subject} shared id {ids_done[int(idx)]}\nactual | VGG-DIP", fontsize=8)
    plt.savefig(preview_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("saved:", preview_path)
    return row


if "dip_manifest" not in globals():
    with open(f"{OUTPUT_DIR}/dip_prediction_manifest_{DIP_OUTPUT_TAG}.json") as f:
        dip_manifest = json.load(f)

eval_rows = [eval_dip_subject(subject, dip_manifest[subject]) for subject in FINAL_EVAL_SUBJECTS]
eval_summary = pd.DataFrame(eval_rows)
EVAL_SUMMARY_PATH = f"{OUTPUT_DIR}/final_dip_eval_summary_{DIP_OUTPUT_TAG}.csv"
eval_summary.to_csv(EVAL_SUMMARY_PATH, index=False)
print(eval_summary)
print("saved summary:", EVAL_SUMMARY_PATH)
